<a href="https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/Updated-Prompt/mnps_prompt_reliability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<!-- Update the link above to point at your repo/branch/notebook if needed. -->

# MNPS Prompt Notebook — Ground Truth–Guided Evaluation (Predictions CSV)

**Purpose.** Use the **Ground Truth Masterfile** as few-shot exemplars to guide evaluation of a **Sample File**, then save the final model output as **`predictions.csv`**.  
All run artifacts are saved locally to `/content/run_artifacts` and exported to Google Drive at `/content/drive/My Drive/Colab Notebooks/Run Results/RUN_<timestamp>`.

**Before you run in Colab:**
1) Upload your CSVs to `/content/`:
   - `Ground Truth Masterfile.csv`
   - `Sample File.csv`
2) Set your `OPENAI_API_KEY` in the session:
   ```python
   import os; os.environ['OPENAI_API_KEY'] = 'sk-...'
   ```


In [ ]:
# 0) Optional installs (Colab)
# !pip install -q openai>=1.40 scikit-learn pydantic

In [ ]:
# 1) Imports & core config
import os, json, time
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# --- FILE PATHS ---
GROUND_TRUTH_CSV = "/content/Ground Truth Masterfile.csv"
SAMPLE_CSV       = "/content/Sample File.csv"

# Directory for local run artifacts
RUN_LOCAL_DIR = "/content/run_artifacts"
Path(RUN_LOCAL_DIR).mkdir(parents=True, exist_ok=True)

# Optional: pin a run id/timestamp for traceability
import time
RUN_ID = time.strftime("%Y%m%d_%H%M%S")

print({
    'GROUND_TRUTH_CSV': GROUND_TRUTH_CSV,
    'SAMPLE_CSV': SAMPLE_CSV,
    'RUN_LOCAL_DIR': RUN_LOCAL_DIR,
    'RUN_ID': RUN_ID
})

In [ ]:
# 2) Ground Truth + Sample loading and exemplar index
TEXT_COLS = [
    "Position Summary",
    "Essential Functions",
    "Education",
    "Work Experience",
    "Licenses and Certifications",
    "Knowledge, Skills and Abilities",
]

LABEL_COLS = [
    "New Job Title", "Major Role Group", "Minor Sub-Group",
    "Major Role", "Minor Role",
    "MNPS Title"
]

def coalesce_cols(df, cols):
    return [c for c in cols if c in df.columns]


def make_text_blob(row, cols):
    parts = []
    for c in cols:
        val = str(row.get(c, "") or "").strip()
        if val and val.lower() not in {"nan", "none"}:
            parts.append(f"{c}: {val}")
    return "\n".join(parts)

print("Loading CSVs…")
gt = pd.read_csv(GROUND_TRUTH_CSV)
sample = pd.read_csv(SAMPLE_CSV)

text_cols_gt      = coalesce_cols(gt, TEXT_COLS)
text_cols_sample  = coalesce_cols(sample, TEXT_COLS)
label_cols_gt     = coalesce_cols(gt, LABEL_COLS)

# Build blobs
gt["__blob__"] = gt.apply(lambda r: make_text_blob(r, text_cols_gt), axis=1)
sample["__blob__"] = sample.apply(lambda r: make_text_blob(r, text_cols_sample), axis=1)

# TF-IDF fit on Ground Truth universe
vec = TfidfVectorizer(min_df=2, ngram_range=(1,2), stop_words="english")
gt_matrix = vec.fit_transform(gt["__blob__"].fillna(""))


def get_topk_exemplars(query_text, k=5):
    if not query_text.strip():
        return gt.iloc[:0]
    q = vec.transform([query_text])
    sims = cosine_similarity(q, gt_matrix)[0]
    topk_idx = np.argsort(-sims)[:k]
    out = gt.iloc[topk_idx].copy()
    out["__sim__"] = sims[topk_idx]
    return out


def format_exemplars_for_prompt(df_ex):
    exemplars = []
    for _, r in df_ex.iterrows():
        label_bits = []
        for c in label_cols_gt:
            if c in r and pd.notna(r[c]) and str(r[c]).strip():
                label_bits.append(f"{c}: {r[c]}")
        label_text = "; ".join(label_bits) if label_bits else "Labels: (not available)"
        blob = r["__blob__"][:1200]
        exemplars.append(f"- EXAMPLE\n{label_text}\nTEXT\n{blob}\n")
    return "\n".join(exemplars) if len(exemplars) else "None"

print("Ground Truth rows:", len(gt))
print("Sample rows:", len(sample))

In [ ]:
# 3) Schema for structured output
from typing import List, Optional
from pydantic import BaseModel

class JobClassification(BaseModel):
    new_job_title: str
    major_role_group: str
    minor_sub_group: Optional[str] = None
    grouping_justification: str

class JobClassificationTable(BaseModel):
    job_classification_table: List[JobClassification]

print("Schema ready.")

In [ ]:
# 4) System prompt rules (Ground Truth-aware)
CLASSIFIER_SYSTEM_PROMPT = """\
You are an MNPS job classification assistant. Classify a single job description into:
- New Job Title (format: "[Function] [Role] [Level]" e.g., "Collections Specialist II")
- Major Role Group (one of: Specialist, Analyst, Director, Manager, Technician, Coordinator)
- Minor Sub-Group (I, II, III, IV; use the closest level; omit 'IV' if inappropriate)
- Grouping Justification (brief, cites signals across Position Summary, Essential Functions, Education, Experience, Licenses/Certifications, KSAs)

CRITICAL RULES:
- Focus on **what the job does** (functions, scope, decision latitude, supervision, consequences of error).
- **Do NOT** overweight literal job-title strings that appear in Position Summary/Essential Functions; treat them as weak hints.
- Heavily weight licensure requirements and scope of responsibility when present.
- Prefer consistency with verified MNPS decisions shown in the EXEMPLARS block.
- If unsure between adjacent levels (e.g., II vs III), choose the more conservative (lower) level and explain why.

Return structured output per the provided schema.
"""
print("Classifier system prompt set.")

In [ ]:
# 5) OpenAI client
from openai import OpenAI
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

if not os.environ.get("OPENAI_API_KEY"):
    print("WARNING: OPENAI_API_KEY not set in environment; set it before running the batch.")
else:
    print("OpenAI API key detected.")

In [ ]:
# 6) Batch run that writes predictions.csv

def classify_one_record(row, k_ex=5):
    ex_df = get_topk_exemplars(row["__blob__"], k=k_ex)
    exemplars_block = format_exemplars_for_prompt(ex_df)

    job_title_original = str(row.get("Job Description Name", "") or "").strip()
    user_blob = f"""\
SAMPLE (to classify)
Original Job Title: {job_title_original or "(unknown)"}

TEXT
{row["__blob__"]}

EXEMPLARS (verified MNPS decisions from Ground Truth; use for guidance)
{exemplars_block}
"""

    messages = [
        {"role": "developer", "content": CLASSIFIER_SYSTEM_PROMPT},
        {"role": "user", "content": user_blob}
    ]

    resp = client.beta.chat.completions.parse(
        model="gpt-4o",
        messages=messages,
        temperature=0.2,
        max_tokens=900,
        response_format=JobClassificationTable
    )

    parsed = resp.choices[0].message.parsed
    jc = parsed.job_classification_table[0]
    out = {
        "run_id": RUN_ID,
        "job_title_original": job_title_original,
        "new_job_title": jc.new_job_title,
        "major_role_group": jc.major_role_group,
        "minor_sub_group": jc.minor_sub_group,
        "grouping_justification": jc.grouping_justification,
    }
    return out

# ---- Run the batch and write predictions.csv ----
results = []
for i, row in sample.iterrows():
    try:
        results.append(classify_one_record(row, k_ex=5))
    except Exception as e:
        results.append({
            "run_id": RUN_ID,
            "job_title_original": str(row.get("Job Description Name", "")),
            "error": str(e),
        })

pred_df = pd.DataFrame(results)

# Write to canonical predictions filename (local & top-level for easy pickup)
predictions_local = f"{RUN_LOCAL_DIR}/predictions.csv"
pred_df.to_csv(predictions_local, index=False)

# Also write a copy in /content (some external programs expect it there)
predictions_top = "/content/predictions.csv"
pred_df.to_csv(predictions_top, index=False)

print("Wrote:")
print(" -", predictions_local)
print(" -", predictions_top)

pred_df.head()

In [ ]:
# 7) Optional — Adjudication sheet scaffold (kept, but predictions.csv is your required output)
adj_cols = [
    "run_id","job_title_original","new_job_title",
    "major_role_group","minor_sub_group","grouping_justification",
    "review_correct_title (Y/N)","review_correct_major (Y/N)","review_correct_minor (Y/N)",
    "review_notes"
]

adj = pred_df.reindex(columns=[c for c in adj_cols if c in pred_df.columns] +
                                 [c for c in adj_cols if c not in pred_df.columns])
adj_path = f"{RUN_LOCAL_DIR}/MNPS_Adjudication_Sheet_{RUN_ID}.csv"
adj.to_csv(adj_path, index=False)
print(f"Wrote: {adj_path}")

In [ ]:
# 8) Export artifacts (including predictions.csv) to Google Drive
from google.colab import drive
import glob, shutil

drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/My Drive/Colab Notebooks/Run Results"
run_dir_drive = f"{DRIVE_ROOT}/RUN_{RUN_ID}"
Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
Path(run_dir_drive).mkdir(parents=True, exist_ok=True)

# Copy run folder contents (includes predictions.csv)
for p in Path(RUN_LOCAL_DIR).glob("*"):
    try:
        shutil.copy2(str(p), run_dir_drive)
    except Exception as e:
        print(f"Skip {p}: {e}")

# Also copy the convenient /content/predictions.csv
try:
    shutil.copy2("/content/predictions.csv", run_dir_drive)
except Exception as e:
    print(f"Skip /content/predictions.csv: {e}")

print(f"Copied artifacts to: {run_dir_drive}")

---
**Output contract**
- The file your downstream program should read is **`predictions.csv`**. It is written to:
  - `/content/run_artifacts/predictions.csv` (canonical local path)
  - `/content/predictions.csv` (convenience copy)
  - `/content/drive/My Drive/Colab Notebooks/Run Results/RUN_<timestamp>/predictions.csv` (exported copy)

If you want to change columns in `predictions.csv`, edit the `out = { ... }` dictionary in cell **6**.
